# FASE 4 · ETL Y CARGA DE DATOS EN MYSQL

En esta fase se realiza la integración del dataset sometido a limpieza y transformación en la fase 2 en una base de datos relacional utilizando MySQL Workbench y SQLAlchemy.

El objetivo es trasladar el archivo generado tras la fase de transformación a un entorno SQL, de forma que los datos queden almacenados en una tabla estructurada y puedan ser consultados posteriormente.

A lo largo de esta fase se realiza:

- la importación de las librerías necesarias,
- la configuración de la conexión con MySQL,
- la carga del dataset limpio,
- la creación de la tabla en la base de datos,
- y la asignación de una clave primaria para asegurar la integridad de los registros.

Esta fase permite conectar el trabajo realizado en Python con un entorno de almacenamiento estructurado.

## Instalación de dependencias

Para ejecutar esta fase correctamente, es necesario disponer de las siguientes librerías:

- sqlalchemy  
- pymysql  

Si no están instaladas en tu entorno, puedes instalarlas ejecutando la siguiente celda.

En caso de que ya estén instaladas, esta celda puede omitirse.

In [1]:
# Instalación de dependencias (ejecutar solo si es necesario)

!pip install sqlalchemy pymysql cryptography


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importación de librerías

import pandas as pd
from sqlalchemy import create_engine, text

## Configuración de la conexión a MySQL

A continuación, se define la conexión con MySQL mediante SQLAlchemy.

Primero se establece una conexión al servidor MySQL para crear la base de datos si no existe. Después, se realiza la conexión al esquema donde se cargará la tabla final. En este caso, la base de datos utilizada es `ABC_Corporation`.

In [3]:
# Configuración de la conexión a MySQL

USER = "root"
PASS = "AlumnaAdalab"
HOST = "127.0.0.1"
DB_NAME = "ABC_Corporation"

# Primero conectamos al servidor MySQL sin indicar base de datos
engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}")

# Creamos la base de datos si no existe
with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))

# Ahora sí nos conectamos a la base de datos
engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}/{DB_NAME}")

## Carga del dataset limpio

En esta celda se carga el archivo generado tras la Fase 2, que contiene los datos ya transformados y preparados para su almacenamiento.

Se visualizan las primeras filas para comprobar que la lectura se ha realizado correctamente antes de subir la información a MySQL.

In [4]:
# Carga del dataset limpio

df_hr = pd.read_csv("datos_limpios.csv")

# Visualización rápida para comprobar que la carga se ha realizado correctamente
df_hr.head()

,Age,Attrition,Business_Travel,Department,Distance_From_Home,Education,Education_Field,Employee_Number,Environment_Satisfaction,Gender,...,Performance_Rating,Relationship_Satisfaction,Stock_Option_Level,Total_Working_Years,Training_Times_Last_Year,Work_Life_Balance,Years_At_Company,Years_In_Current_Role,Years_Since_Last_Promotion,Years_With_Curr_Manager
0,41,Yes,Travel Rarely,Sales,1,2,Life Sciences,1,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel Frequently,Research & Development,8,1,Life Sciences,2,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel Rarely,Research & Development,2,2,Other,4,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel Frequently,Research & Development,3,4,Life Sciences,5,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel Rarely,Research & Development,2,1,Medical,7,1,Male,...,3,4,1,6,3,3,2,2,2,2


## Carga del DataFrame en MySQL

Una vez cargado el dataset limpio, se sube a la base de datos mediante el método `to_sql()` de pandas.

Este proceso crea automáticamente la tabla `employees_attrition` dentro del esquema `ABC_Corporation`.

Antes de asignar la clave primaria, se eliminan posibles valores nulos o duplicados en la columna `Employee_Number` para garantizar la integridad de los registros.

Posteriormente, se ejecuta una instrucción SQL para asignar la clave primaria a dicha columna, asegurando así la unicidad de los registros.

In [5]:
# Carga del DataFrame en MySQL
# Esto crea automáticamente la tabla employees_attrition en la base de datos

df_hr.to_sql("employees_attrition", con=engine, if_exists="replace", index=False)

# Comprobación previa para evitar errores al asignar la clave primaria
df_hr = df_hr.dropna(subset=["Employee_Number"])
df_hr = df_hr.drop_duplicates(subset=["Employee_Number"])

df_hr.to_sql("employees_attrition", con=engine, if_exists="replace", index=False)

# Asignación de la clave primaria una vez creada la tabla
with engine.begin() as con:
    con.execute(text("""
        ALTER TABLE employees_attrition
        MODIFY Employee_Number INT NOT NULL"""))
    con.execute(text("""
        ALTER TABLE employees_attrition
        ADD PRIMARY KEY (Employee_Number)"""))

print("¡Listo! Base de datos, tabla y clave primaria creadas correctamente.")

¡Listo! Base de datos, tabla y clave primaria creadas correctamente.


### Conclusión

Esta fase permite trasladar el dataset limpio a un entorno relacional, conectando el trabajo realizado en Python con MySQL Workbench.

Como resultado, los datos quedan almacenados en el esquema `ABC_Corporation`, dentro de la tabla `employees_attrition`, listos para su consulta y reutilización en futuros análisis.

De este modo, el proyecto no solo incluye exploración, transformación y análisis de datos, sino también una primera integración en base de datos.